# EMO1 - series areales
***

***Autor:** Jesús Casado Rodríguez*<br>
***Fecha:** 27-03-2026*<br>

**Introducción:**<br>
En este _notebook_ se calculan las series meteorológicas areales de las cuencas de CAMELS-ES a partir de los datos meteorológicos de EMO1, los datos de entrada del sistema EFASv5. 

Las series meteorológicas generadas serán los datos dinámicos de entrada para el LSTM que replique el modelo hidrológico LISFLOOD, el usado en EFASv5.

**Por hacer:**<br>
* [x] Las series empiezan el 1 de enero de 1990 y terminan el 1 de enero del 2020. Deberían empezar el 1 de octubre de 2021 y terminar el 30 de septiembre de 2020.

These are the steps in `catchstats:main`:

```Python
    maps = read_inputmaps(args.input)
    masks = read_masks(args.mask)
    weight = read_pixarea(args.area) if args.area is not None else None
    catchment_statistics(maps, masks, args.statistic, weight=weight, output=args.output, overwrite=args.overwrite)
```

In [2]:
# import glob
# import matplotlib.pyplot as plt
import geopandas as gpd
from tqdm.notebook import tqdm
import xarray as xr
# import cartopy.crs as ccrs
# import cartopy.feature as cf
# from pathlib import Path

#from funciones import polygon_statistics, read_static_map

from camels_es.config import Config
from camels_es.catchstats import read_data, read_pixarea, catchment_statistics

In [3]:
# load configuration file
cfg = Config('../config_CAMELS.yml')

In [4]:
# load basins shapefile
basins = gpd.read_file(
    cfg.path_dataset / 'preprocessing' / 'basins' / 'output' / 'stations_basins_1min.geojson'
    ).set_index('ID')

In [5]:
# # plot basins
# proj = ccrs.PlateCarree()
# fig, ax = plt.subplots(subplot_kw={'projection': proj})
# ax.add_feature(cf.NaturalEarthFeature('physical', 'land', '50m', edgecolor=None, facecolor='lightgray'), zorder=0)
# ax.set_extent([-9.5, 3.5, 36, 44.5], crs=proj)
# basins.plot(ax=ax, facecolor='none', edgecolor='w', linewidth=0.3);
# ax.axis('off');

In [9]:
# load pixel area to weigh the statistics
pixarea = read_pixarea(cfg.path_efas / 'maps' / 'pixarea_iberian_01min.nc')
pixarea = pixarea.rio.write_crs(cfg.crs)

In [8]:
# mload meteorological data
zarr_store = cfg.path_meteo / 'EMO1_1990-2022.zarr'
data = read_data(zarr_store, engine='zarr')
data = data.rio.write_crs(cfg.crs)

print(f"{data.nbytes / 1e9:.2f} GB")

67.95 GB


In [ ]:
# compute statistics
results = catchment_statistics(
    data=data,
    basins=basins,
    statistic='mean',
    weight=pixarea,
    decimals=1,
    output=cfg.path_dataset / 'preprocessing' / 'catchstats' / 'meteo'
)

Basins:   0%|          | 0/833 [00:00<?, ?it/s]